# 01 画像の基礎 — 画像は「数値の配列」

> **このノートの使い方**：セルを選んで **Shift + Enter** で実行します。上から順に進めてください。
> 緑の **✅ チェックポイント** が出たら次へ。
> 🤖 **困ったら ChatGPT に聞く**（エラーは*全文*を貼る／コードの意味も気軽に質問）。TA への挙手もOK。

## 1-1. 題材の画像を用意
まずカメラで 1 枚撮って、変数 `img` に入れます（以降このノートで使い回します）。

In [ ]:
import cv2, numpy as np, time
import matplotlib.pyplot as plt
%matplotlib inline
from picamera2 import Picamera2

def show(bgr, title='', size=(6,4), gray=False):
    '''OpenCV の BGR 画像をノートにインライン表示するヘルパー'''
    plt.figure(figsize=size)
    if gray:
        plt.imshow(bgr, cmap='gray')
    else:
        plt.imshow(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
    plt.axis('off');
    if title: plt.title(title)
    plt.show()

picam = Picamera2()
picam.configure(picam.create_preview_configuration(main={'size':(640,480),'format':'RGB888'}))
picam.start(); time.sleep(1)
img = picam.capture_array(); picam.close()
show(img, '題材の画像 img')

## 1-2. 画像の正体を見る
画像は `(高さ, 幅, 3)` の **numpy 配列**。1 画素は 3 つの数値（各 0–255）です。

In [ ]:
print('shape:', img.shape)        # (高さ, 幅, チャンネル数)
print('dtype:', img.dtype)        # uint8 = 0..255
print('左上の画素 (B,G,R):', img[0, 0])

✅ **チェックポイント**：`shape: (480, 640, 3)` のように表示されれば OK。

> **大事なクセ**：OpenCV の色順は **BGR**（青・緑・赤）。次でその意味を目で確かめます。

## 1-3. BGR を体感する
OpenCV の配列を **RGB と思って**表示すると、赤と青が入れ替わって見えます。

In [ ]:
plt.figure(figsize=(10,4))
plt.subplot(1,2,1); plt.imshow(img); plt.axis('off')
plt.title('間違い：BGR配列をそのまま表示（色が変）')
plt.subplot(1,2,2); plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); plt.axis('off')
plt.title('正解：BGR→RGB に変換して表示')
plt.show()

→ 左が「色が変」になります。**matplotlib や PIL に渡すときだけ** `cv2.cvtColor(img, cv2.COLOR_BGR2RGB)` を通すと覚えましょう。

## 1-4. リサイズ・トリミング・描画

In [ ]:
small = cv2.resize(img, (320, 240))         # サイズは (幅, 高さ)
roi   = img[100:300, 200:440]                # スライスで切り抜き [y1:y2, x1:x2]
canvas = img.copy()
cv2.rectangle(canvas, (200,100), (440,300), (0,255,0), 2)  # 緑の枠 (色は BGR)
cv2.putText(canvas, 'ROI', (200,90), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0,0,255), 2)
show(roi, '切り抜き roi'); show(canvas, '枠と文字を描画')

## 1-5. 色空間：グレースケールと HSV
目的に応じて色の表し方を変えると、後の処理が簡単になります。

In [ ]:
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
show(gray, 'グレースケール（明るさだけ）', gray=True)

hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
mask = cv2.inRange(hsv, (35,80,80), (85,255,255))   # 緑のあたり (H は 0..179)
show(mask, '緑を白で抜き出したマスク', gray=True)
print('緑と判定した画素数:', int((mask>0).sum()))

✅ **チェックポイント**：グレー画像と、緑だけ白いマスク画像が表示されれば OK。

## 1-6. やってみよう
- `inRange` の範囲を **青**（H を 100〜130）に変えて、青い物だけ抜き出してみる
- 切り抜く範囲 `[y1:y2, x1:x2]` を変えて、好きな場所を切り出す

> 🤖 **ChatGPT に聞いてみよう**
> 「OpenCV(Python)で、HSV を使って**青い物だけ**白く抜き出すコードを書いて。OpenCV の H は 0–179 に注意」と聞いて、出たコードをこの下のセルで試そう。

In [ ]:
# ここに自分のコードを書いて Shift+Enter